# Kaggle smoke test for PoRT LLM Unlearning Experiment

This notebook is intentionally small. It does not load large LLM checkpoints or reproduce the full paper results yet.

What it checks:
- the repo is cloned from GitHub on Kaggle, or the local checkout is used outside Kaggle;
- Python source files compile;
- obvious placeholder paths and package import issues are reported;
- a few TOFU and WMDP dataset samples can be loaded;
- lightweight evaluator logic runs on synthetic/sample data.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import re
import subprocess
import sys
import traceback


REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'llm-unlearn-eco' / 'eco').exists() and (path / 'dataset').exists()


def refresh_existing_clone(path):
    path = Path(path)
    if (path / '.git').exists():
        print(f'Refreshing existing clone: {path}')
        subprocess.check_call(['git', '-C', str(path), 'pull', '--ff-only'])


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            refresh_existing_clone(target)
            return target.resolve()
        if target.exists():
            raise RuntimeError(
                f'{target} exists but does not look like this repo. Remove or rename it before rerunning this notebook.'
            )
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        if not has_project_layout(target):
            raise FileNotFoundError(f'Cloned repo does not contain the expected project layout: {target}')
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        print(f'Using local checkout: {local_root}')
        return local_root

    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        print(f'Using existing cloned repository: {target}')
        return target.resolve()
    if target.exists():
        raise RuntimeError(
            f'{target} exists but does not look like this repo. Remove or rename it before rerunning this notebook.'
        )
    print(f'Cloning {REPO_URL} into {target}')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    if not has_project_layout(target):
        raise FileNotFoundError(f'Cloned repo does not contain the expected project layout: {target}')
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
TOFU_DIR = PROJECT_ROOT / 'dataset' / 'TOFU' / 'original'
WMDP_DIR = PROJECT_ROOT / 'dataset' / 'WMDP' / 'original'
if str(ECO_ROOT) not in sys.path:
    sys.path.insert(0, str(ECO_ROOT))

logic_warnings = []
print(f'Project root: {PROJECT_ROOT}')
print(f'ECO root:     {ECO_ROOT}')
print(f'TOFU data:    {TOFU_DIR}')
print(f'WMDP data:    {WMDP_DIR}')


In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'rouge_score': 'rouge-score>=0.1.2',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing smoke-test packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required smoke-test packages are already available.')


## Source and package checks

The compile check is fatal because syntax errors block any run. Package-level imports are reported but not fatal here, because this smoke notebook also uses isolated imports to test the files that are present in this repo snapshot.


In [ ]:
source_roots = [
    ECO_ROOT / 'eco',
    ECO_ROOT / 'scripts',
    PROJECT_ROOT / 'PoRT_pipeline',
    PROJECT_ROOT / 'post_classifier',
]
python_files = sorted({path for root in source_roots if root.exists() for path in root.rglob('*.py')})

syntax_failures = []
for path in python_files:
    try:
        compile(path.read_text(encoding='utf-8'), str(path), 'exec')
    except Exception as exc:
        syntax_failures.append({'file': str(path.relative_to(PROJECT_ROOT)), 'error': repr(exc)})

print(f'Compiled {len(python_files)} Python files.')
if syntax_failures:
    print(json.dumps(syntax_failures, indent=2))
    raise SyntaxError('Smoke test found Python syntax/compile failures.')
print('Syntax compile check passed.')


In [ ]:
placeholder_hits = {}
placeholder_pattern = re.compile(r'<[A-Z0-9_/-]+>')
for path in python_files:
    hits = sorted(set(placeholder_pattern.findall(path.read_text(encoding='utf-8'))))
    if hits:
        placeholder_hits[str(path.relative_to(PROJECT_ROOT))] = hits

if placeholder_hits:
    logic_warnings.append('Placeholder paths still exist; full reproduction needs concrete Kaggle/local paths.')
    print('Placeholder paths found:')
    print(json.dumps(placeholder_hits, indent=2))
else:
    print('No placeholder paths found in Python files.')

wmdp_source = (ECO_ROOT / 'eco' / 'dataset' / 'wmdp.py').read_text(encoding='utf-8')
missing_wmdp_symbols = [symbol for symbol in ['class WMDP', 'class WMDPBio', 'class WMDPChem', 'class WMDPCyber'] if symbol not in wmdp_source]
if missing_wmdp_symbols:
    message = 'WMDP dataset module is missing expected class definitions: ' + ', '.join(missing_wmdp_symbols)
    logic_warnings.append(message)
    print('WARNING:', message)
else:
    print('WMDP dataset class definitions are present.')


In [ ]:
modules_to_try = ['eco.dataset', 'eco.evaluator', 'eco.inference', 'eco.utils']
package_import_results = {}
for module_name in modules_to_try:
    try:
        importlib.import_module(module_name)
        package_import_results[module_name] = 'OK'
    except Exception as exc:
        package_import_results[module_name] = f'{type(exc).__name__}: {exc}'

print(json.dumps(package_import_results, indent=2))
failed_package_imports = {k: v for k, v in package_import_results.items() if v != 'OK'}
if failed_package_imports:
    logic_warnings.append('Package-level imports failed; inspect package_import_results before full evaluation.')


## Isolated imports for present modules

This avoids failing on missing optional dataset/evaluator modules while still exercising the TOFU files and lightweight evaluator files that are present.


In [ ]:
import importlib.util
import types


def install_namespace_package(name, package_path):
    module = types.ModuleType(name)
    module.__path__ = [str(package_path)]
    sys.modules[name] = module
    return module


def load_module_from_path(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


for name in list(sys.modules):
    if name == 'eco' or name.startswith('eco.'):
        sys.modules.pop(name, None)

install_namespace_package('eco', ECO_ROOT / 'eco')
install_namespace_package('eco.dataset', ECO_ROOT / 'eco' / 'dataset')
install_namespace_package('eco.evaluator', ECO_ROOT / 'eco' / 'evaluator')
install_namespace_package('eco.attack', ECO_ROOT / 'eco' / 'attack')

base_mod = load_module_from_path('eco.dataset.base', ECO_ROOT / 'eco' / 'dataset' / 'base.py')
tofu_mod = load_module_from_path('eco.dataset.tofu', ECO_ROOT / 'eco' / 'dataset' / 'tofu.py')
rouge_recall_mod = load_module_from_path('eco.evaluator.rouge_recall', ECO_ROOT / 'eco' / 'evaluator' / 'rouge_recall.py')
choice_logit_mod = load_module_from_path('eco.evaluator.choice_by_top_logit', ECO_ROOT / 'eco' / 'evaluator' / 'choice_by_top_logit.py')

TOFU = tofu_mod.TOFU
TOFUPerturbed = tofu_mod.TOFUPerturbed
ROUGERecall = rouge_recall_mod.ROUGERecall
ChoiceByTopLogit = choice_logit_mod.ChoiceByTopLogit
print('Isolated imports loaded: TOFU, TOFUPerturbed, ROUGERecall, ChoiceByTopLogit')


## TOFU smoke tests


In [ ]:
required_tofu_files = [
    'forget01.json', 'forget05.json', 'forget10.json',
    'retain90.json', 'retain95.json', 'retain99.json',
    'real_authors.json', 'world_facts.json',
    'forget01_perturbed.json', 'retain_perturbed.json',
]
missing_tofu_files = [name for name in required_tofu_files if not (TOFU_DIR / name).exists()]
assert not missing_tofu_files, f'Missing TOFU files: {missing_tofu_files}'

tofu_file_counts = {}
for name in required_tofu_files:
    data = json.loads((TOFU_DIR / name).read_text(encoding='utf-8'))
    assert isinstance(data, list) and data, f'{name} must be a non-empty JSON list.'
    assert {'question', 'answer'}.issubset(data[0]), f'{name} first row must contain question and answer.'
    tofu_file_counts[name] = len(data)

print('TOFU file counts:')
print(json.dumps(tofu_file_counts, indent=2))


In [ ]:
formatting_tokens = {
    'prompt_prefix': 'Question: ',
    'prompt_suffix': '\nAnswer:',
    'answer_prefix': ' ',
    'answer_suffix': '',
}

tofu = TOFU(formatting_tokens=formatting_tokens, eos_token='')
tofu.path = str(TOFU_DIR)
tofu.download()
tofu_counts = {name: len(tofu.dataset[name]) for name in tofu.subsets}
assert all(count > 0 for count in tofu_counts.values()), tofu_counts

eval_dataset = tofu.load_dataset_for_eval('forget01')
assert {'prompt', 'answer', 'prompt_formatted'}.issubset(set(eval_dataset.column_names)), eval_dataset.column_names
sample_n = min(3, len(eval_dataset))
tofu_eval_samples = [eval_dataset[i] for i in range(sample_n)]
assert all(row['prompt_formatted'].startswith('Question: ') for row in tofu_eval_samples)
assert all(row['answer'].startswith(' ') for row in tofu_eval_samples)

batched_eval = tofu.load_dataset_for_eval('forget01', load_in_batch=True, batch_size=2)
assert batched_eval and len(batched_eval[0]['prompt']) <= 2

classification_dataset = tofu.load_dataset_for_classification('forget01', use_val=True)
classification_counts = {split: len(classification_dataset[split]) for split in classification_dataset.keys()}
train_labels = set(classification_dataset['train']['label'])
assert train_labels.issubset({0, 1}) and train_labels, train_labels

print('TOFU subset counts:', tofu_counts)
print('TOFU classification counts:', classification_counts)
print('TOFU eval samples:')
print(json.dumps(tofu_eval_samples, indent=2)[:2000])


In [ ]:
tofu_perturbed = TOFUPerturbed(formatting_tokens, '')
tofu_perturbed.path = str(TOFU_DIR)
perturbed_eval = tofu_perturbed.load_dataset_for_eval('forget01_perturbed')
assert {'prompt', 'answer', 'perturbed_answer', 'choices', 'prompt_formatted'}.issubset(set(perturbed_eval.column_names)), perturbed_eval.column_names
perturbed_sample = perturbed_eval[0]
assert isinstance(perturbed_sample['choices'], list) and len(perturbed_sample['choices']) >= 2

if not perturbed_sample['prompt_formatted'].startswith('Question: '):
    logic_warnings.append('TOFUPerturbed does not apply formatting_tokens with the current constructor call path.')

print('TOFUPerturbed first sample:')
print(json.dumps({
    'prompt_formatted': perturbed_sample['prompt_formatted'],
    'answer': perturbed_sample['answer'],
    'num_choices': len(perturbed_sample['choices']),
}, indent=2)[:2000])


## WMDP parquet sample smoke test


In [ ]:
import pandas as pd

wmdp_samples = {}
for subject in ['bio', 'chem', 'cyber']:
    parquet_path = WMDP_DIR / f'wmdp-{subject}' / 'test-00000-of-00001.parquet'
    assert parquet_path.exists(), f'Missing WMDP parquet: {parquet_path}'
    df = pd.read_parquet(parquet_path)
    assert len(df) > 0, f'{parquet_path} is empty.'
    assert {'question', 'choices'}.issubset(set(df.columns)), df.columns.tolist()
    answer_col = 'answer' if 'answer' in df.columns else 'correct_answer'
    assert answer_col in df.columns, df.columns.tolist()
    sample_df = df.head(3).copy()
    wmdp_samples[subject] = {
        'rows': int(len(df)),
        'columns': df.columns.tolist(),
        'sample': sample_df.to_dict(orient='records'),
    }

print('WMDP sample summary:')
print(json.dumps(wmdp_samples, indent=2, default=str)[:3000])


## Evaluator smoke tests without loading an LLM


In [ ]:
rouge = ROUGERecall(mode='rougeL')
rouge_scores = rouge.evaluate(
    ['The model should return the exact reference answer.'],
    ['The model should return the exact reference answer.'],
)
assert rouge_scores[0] > 0.99, rouge_scores
print('ROUGERecall smoke score:', rouge_scores[0])


In [ ]:
try:
    import torch

    class TensorBatch(dict):
        def __getattr__(self, name):
            return self[name]

        def to(self, device):
            return self


    class FakeTokenizer:
        padding_side = 'right'
        choice_to_id = {'A': 10, 'B': 11, 'C': 12, 'D': 13}

        def __call__(self, texts, padding=None, return_tensors=None, add_special_tokens=True, truncation=False, max_length=None):
            if isinstance(texts, str):
                texts = [texts]
            if texts and all(text in self.choice_to_id for text in texts):
                input_ids = [[self.choice_to_id[text]] for text in texts]
                attention_mask = [[1] for _ in texts]
            else:
                lengths = [max(1, len(text.split())) for text in texts]
                max_len = max(lengths)
                input_ids = [[1] * length + [0] * (max_len - length) for length in lengths]
                attention_mask = [[1] * length + [0] * (max_len - length) for length in lengths]
            return TensorBatch({
                'input_ids': torch.tensor(input_ids, dtype=torch.long),
                'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            })


    class FakeOutput:
        def __init__(self, logits):
            self.logits = logits


    class FakeModel:
        device = 'cpu'

        def __call__(self, input_ids, attention_mask, prompts=None, answers=None):
            batch_size, seq_len = input_ids.shape
            logits = torch.zeros(batch_size, seq_len, 32)
            winners = [11, 13]
            for i in range(batch_size):
                prompt_end = int(attention_mask[i].sum().item()) - 1
                logits[i, prompt_end, winners[i]] = 10.0
            return FakeOutput(logits)


    fake_prompts = ['short prompt', 'another prompt']
    fake_choices = [['A', 'B', 'C', 'D'], ['A', 'B', 'C', 'D']]
    fake_predictions = ChoiceByTopLogit().evaluate(fake_prompts, fake_choices, FakeModel(), FakeTokenizer())
    assert fake_predictions == [1, 3], fake_predictions
    print('ChoiceByTopLogit fake-model predictions:', fake_predictions)
except ModuleNotFoundError as exc:
    logic_warnings.append(f'Skipped ChoiceByTopLogit fake-model smoke because {exc.name} is unavailable.')
    print('Skipped ChoiceByTopLogit fake-model smoke:', exc)


## Smoke summary artifact


In [ ]:
output_dir = (Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT) / 'smoke_outputs'
output_dir.mkdir(parents=True, exist_ok=True)

smoke_summary = {
    'project_root': str(PROJECT_ROOT),
    'syntax_files_checked': len(python_files),
    'syntax_failures': syntax_failures,
    'placeholder_hits': placeholder_hits,
    'package_import_results': package_import_results,
    'logic_warnings': logic_warnings,
    'tofu_file_counts': tofu_file_counts,
    'tofu_counts': tofu_counts,
    'tofu_classification_counts': classification_counts,
    'tofu_eval_sample_count': len(tofu_eval_samples),
    'wmdp_counts': {subject: data['rows'] for subject, data in wmdp_samples.items()},
    'rouge_recall_smoke_score': rouge_scores[0],
}

summary_path = output_dir / 'smoke_summary.json'
summary_path.write_text(json.dumps(smoke_summary, indent=2, default=str), encoding='utf-8')

sample_path = output_dir / 'tofu_eval_samples.json'
sample_path.write_text(json.dumps(tofu_eval_samples, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'Wrote summary to: {summary_path}')
print(f'Wrote TOFU samples to: {sample_path}')
print(json.dumps(smoke_summary, indent=2, default=str)[:4000])
print('SMOKE TEST COMPLETED')
